# 03 — Chunking strategies, measured with RAGAS

Companion notebook to **M16 — RAG in production** (slides on RAGAS metrics + abstention).

We pit two chunking strategies against each other on the same corpus:

1. **Naive chunking** — the project's `chunker.chunk_doc` (size 2000, overlap 200, character-level)
2. **Section-aware chunking** — split on markdown headings; sub-split long sections by paragraph

Both indices live in **ChromaDB** as two separate collections. The same answer pipeline runs over both; **RAGAS** scores the outputs side by side.

The augmented QA set in `eval/questions.json` adds 5 in-doc questions and 5 deliberately unanswerable distractors, so we can measure **abstention recall** alongside the four RAGAS metrics.

> The hypothesis worth testing: does keeping each topical thread intact (section-aware) actually beat fixed-size chunks once we measure faithfulness and context recall? Or does the simpler chunker hold its own?


In [ ]:
# Install RAGAS + ChromaDB + langchain pieces if missing. Harmless if already there.
%pip install -q chromadb ragas datasets langchain-openai langchain-text-splitters


In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
import sys, json, re, time
from pathlib import Path

import numpy as np
import pandas as pd

# Add repo root to sys.path so we can import project packages.
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from document_processing.extract import Document
from chunker import chunk_doc
from retriever.retrieve import RetrievalResult
from answerer import answer

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction


In [ ]:
# Paths
MARKDOWN_DIR    = ROOT_DIR / "data" / "markdown"
DOCLING_MD      = MARKDOWN_DIR / "Large Language Models the New Scrum Masters [docling].md"
QUESTIONS_FILE  = ROOT_DIR / "eval" / "questions.json"
CHROMA_DIR      = Path.cwd() / "chroma_db"   # persists next to the notebook

# Knobs
EMB_MODEL_NAME  = "all-MiniLM-L6-v2"
NAIVE_SIZE      = 2000
NAIVE_OVERLAP   = 200
SECTION_MAX     = 3000        # split long sections into ≤ this many chars
TOP_K           = 5

ANSWERER_MODEL  = "gpt-4o-mini"
JUDGE_MODEL     = "gpt-4o"     # different from answerer → no self-grading bias

print("docling markdown :", DOCLING_MD.name, "(exists:", DOCLING_MD.exists(), ")")
print("chroma store     :", CHROMA_DIR)


## 1. Load the docling-converted markdown

The PDF was preprocessed earlier with **docling** and saved under `data/markdown/`. We consume that markdown directly — no re-extraction. The YAML frontmatter at the top tells us the title and authors; the body is what we'll chunk.


In [ ]:
import frontmatter
post = frontmatter.loads(DOCLING_MD.read_text(encoding="utf-8"))

doc = Document(
    source=DOCLING_MD,
    text=post.content,
    metadata={"title": post.get("title"), "authors": post.get("authors") or []},
)
print(f"title    : {doc.metadata['title']}")
print(f"authors  : {doc.metadata['authors']}")
print(f"length   : {len(doc.text):,} characters")
print()
print(doc.text[:400], "...")


## 2. Naive chunking — `chunker.chunk_doc`

Character-window splitter with a fixed `size` and `overlap`. No awareness of headings or paragraphs — it just steps through the text and emits windows.

This is the project's default chunker and the same one driving `eval/run_eval.py`.


In [ ]:
naive_chunks = chunk_doc(doc, size=NAIVE_SIZE, overlap=NAIVE_OVERLAP)

print(f"naive chunks    : {len(naive_chunks)}")
print(f"avg chunk size  : {np.mean([len(c.text) for c in naive_chunks]):.0f} chars")
print(f"sample chunk 0  : {naive_chunks[0].text[:200]!r}…")


## 3. Section-aware chunking — LangChain markdown splitters

Two LangChain splitters chained together:

1. **`MarkdownHeaderTextSplitter`** — split on `#`, `##`, `###` headings. Each output piece carries the heading hierarchy as metadata (`Header 1`, `Header 2`, `Header 3`).
2. **`RecursiveCharacterTextSplitter`** — sub-split any section longer than `SECTION_MAX` so chunk sizes stay comparable to the naive baseline. Splits on paragraph → sentence → word boundaries in order.

The two-stage pattern (heading split → recursive char split) is the canonical LangChain recipe for markdown.


In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

HEADERS_TO_SPLIT_ON = [("#", "H1"), ("##", "H2"), ("###", "H3")]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=HEADERS_TO_SPLIT_ON,
    strip_headers=False,           # keep the heading text inside the chunk
)
char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=SECTION_MAX,
    chunk_overlap=0,               # section-aware: no overlap by design
)

# Stage 1: split on headings → list[Document] with hierarchy in metadata
header_splits = md_splitter.split_text(doc.text)

# Stage 2: sub-split any section that's still too long
section_docs = char_splitter.split_documents(header_splits)

# Flatten to the same shape as naive_chunks (text + path + source) for downstream code.
def _section_path(meta: dict) -> str:
    return " > ".join(meta[k] for k in ("H1", "H2", "H3") if k in meta)

section_pieces = [
    {
        "text":         d.page_content,
        "section_path": _section_path(d.metadata),
        "source":       DOCLING_MD.name,
    }
    for d in section_docs
]

print(f"section chunks  : {len(section_pieces)}")
print(f"avg chunk size  : {np.mean([len(c['text']) for c in section_pieces]):.0f} chars")
print(f"first 3 section paths:")
for c in section_pieces[:3]:
    print(f"  - {c['section_path']!r}")


## 4. Persist both chunk sets in ChromaDB

One persistent client → two collections (`naive_chunks` and `section_chunks`).
Embeddings use the same `all-MiniLM-L6-v2` model for both, so the only variable is the chunking strategy.

Re-runs are cheap: if the collection already has the expected row count, we reuse it. If not (e.g., you tweaked `NAIVE_SIZE`), the collection is rebuilt.


In [ ]:
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
ef = SentenceTransformerEmbeddingFunction(model_name=EMB_MODEL_NAME)

def ensure_collection(name, expected_count, ids, documents, metadatas):
    '''Get-or-rebuild pattern. Rebuilds only when the on-disk count is wrong.'''
    col = client.get_or_create_collection(
        name=name,
        embedding_function=ef,
        metadata={"hnsw:space": "cosine"},
    )
    if col.count() == expected_count:
        print(f"  {name}: reusing {col.count()} existing chunks")
        return col
    if col.count() > 0:
        client.delete_collection(name)
        col = client.create_collection(
            name=name, embedding_function=ef,
            metadata={"hnsw:space": "cosine"},
        )
    print(f"  {name}: indexing {expected_count} chunks ...")
    col.add(ids=ids, documents=documents, metadatas=metadatas)
    return col

# Naive
naive_ids   = [c.id for c in naive_chunks]
naive_docs  = [c.text for c in naive_chunks]
naive_meta  = [{"source": c.source, "index": c.index} for c in naive_chunks]
naive_col   = ensure_collection("naive_chunks", len(naive_chunks),
                                naive_ids, naive_docs, naive_meta)

# Section-aware
section_ids   = [f"section#{i}" for i in range(len(section_pieces))]
section_docs  = [c["text"] for c in section_pieces]
section_meta  = [{"source": c["source"], "section_path": c["section_path"]} for c in section_pieces]
section_col   = ensure_collection("section_chunks", len(section_pieces),
                                  section_ids, section_docs, section_meta)


## 5. Load the augmented QA set

`eval/questions.json` now contains **20 questions**: the original 10 plus 5 new in-doc questions (`mc-006`..`mc-008`, `ow-006`..`ow-007`) and 5 unanswerable distractors (`un-001`..`un-005`).

The unanswerable questions are deliberately about topics the corpus has no opinion on (Tokyo population, Eiffel Tower, mercury, FIFA, speed of light). They probe whether the answerer correctly says *"I don't know"* when retrieval can't help.


In [ ]:
questions = json.loads(QUESTIONS_FILE.read_text(encoding="utf-8"))
qa = pd.DataFrame(questions)
# Promote gold excerpt and gold answer presence
qa['gold_excerpt']    = qa['answer_location'].apply(lambda d: d['excerpt'] if d else None)
qa['is_unanswerable'] = qa['type'].eq('unanswerable')

print(f"total questions    : {len(qa)}")
print(f"unanswerable count : {qa['is_unanswerable'].sum()}")
qa[['id', 'type', 'question', 'answer']].head(20)


## 6. Pipeline — `retrieve → answer` against either collection

One function we'll call twice (once per collection). Retrieval is `collection.query` from ChromaDB; the answerer is the project's own **`answerer.answer`** — we just feed it the ChromaDB hits wrapped as `RetrievalResult` objects.

We override the project's `DEFAULT_PROMPT` with one that **explicitly licences refusal** (`"reply exactly with 'I don't know'"`) — this is what makes the abstention probe well-defined. The model stays at `gpt-4o-mini` (the project default).


In [ ]:
ABSTENTION_PROMPT = '''You are answering a question using ONLY the provided context.

If the context does not contain the answer, reply exactly with "I don't know".
Otherwise, answer concisely.

Question: {query}

Context:
{chunks}
'''

def chromadb_hits_to_results(hits) -> list[RetrievalResult]:
    '''Adapt one ChromaDB `collection.query` response → list[RetrievalResult]
    so the project's `answer()` can consume it unchanged.'''
    ids   = hits['ids'][0]
    docs  = hits['documents'][0]
    metas = hits.get('metadatas', [[{}] * len(ids)])[0] or [{}] * len(ids)
    dists = hits.get('distances', [[None] * len(ids)])[0] or [None] * len(ids)
    return [
        RetrievalResult(
            chunk_id = ids[i],
            source   = (metas[i] or {}).get('source', ''),
            text     = docs[i],
            score    = (1.0 - dists[i]) if dists[i] is not None else 0.0,
        )
        for i in range(len(ids))
    ]

def run_pipeline(collection, qa_df: pd.DataFrame, top_k: int = TOP_K) -> pd.DataFrame:
    '''Return one row per question with the four fields RAGAS needs.'''
    rows = []
    t0 = time.time()
    for q in qa_df.itertuples():
        hits = collection.query(query_texts=[q.question], n_results=top_k)
        results = chromadb_hits_to_results(hits)
        out = answer(
            query  = q.question,
            chunks = results,
            prompt = ABSTENTION_PROMPT,
            model  = ANSWERER_MODEL,
        )
        rows.append({
            "id":                 q.id,
            "type":               q.type,
            "user_input":         q.question,
            "response":           out.text,
            "retrieved_contexts": [r.text for r in results],
            "reference":          q.answer if q.answer is not None else "I don't know",
            "reference_contexts": [q.gold_excerpt] if q.gold_excerpt else [""],
        })
    print(f"  {time.time()-t0:.1f}s wall  |  {len(rows)} queries")
    return pd.DataFrame(rows)

print("running naive pipeline ...")
naive_runs   = run_pipeline(naive_col, qa)
print("running section pipeline ...")
section_runs = run_pipeline(section_col, qa)

naive_runs[['id', 'type', 'response']].head()


## 7. String-matching answer accuracy

The cheapest, most deterministic correctness check: does the **gold answer appear as a substring of the model's response**, after light normalisation? This is the exact pattern used by `eval/run_eval.py` (`normalize` + `answer_hit`) — re-implemented here so the notebook is self-contained.

Brittle on paraphrases (`"three"` vs `"3"`, `"Embry-Riddle"` vs `"Embry-Riddle Aeronautical University"`) but a useful **floor**: any miss here is either a real failure or a paraphrase the next section's `FactualCorrectness` will catch.

We only score the 15 answerable questions — unanswerable ones are graded separately by abstention recall.


In [ ]:
def normalize(s: str) -> str:
    return " ".join(s.lower().split())

def answer_hit(model_answer: str, expected: str) -> bool:
    return normalize(expected) in normalize(model_answer)

def answer_accuracy(runs_df: pd.DataFrame) -> dict:
    answerable = runs_df[runs_df['type'] != 'unanswerable']
    hits = [answer_hit(r.response, r.reference) for r in answerable.itertuples()]
    return {
        'n_answerable': len(answerable),
        'n_hits':       sum(hits),
        'accuracy':     sum(hits) / len(answerable) if len(answerable) else None,
    }

print("naive  :", answer_accuracy(naive_runs))
print("section:", answer_accuracy(section_runs))


In [ ]:
# Per-question hit table — show each chunker's response alongside its hit verdict
# so it's unambiguous which response produced which hit.
def hit_table(runs_df, label):
    df = runs_df[runs_df['type'] != 'unanswerable'].copy()
    df[f'{label}_response'] = df['response']
    df[f'{label}_hit']      = [answer_hit(r.response, r.reference) for r in df.itertuples()]
    return df[['id', 'reference', f'{label}_response', f'{label}_hit']]

tbl = hit_table(naive_runs, 'naive').merge(
    hit_table(section_runs, 'section').drop(columns=['reference']),
    on='id',
)
# Final column order: question id, gold, then naive (response + hit) and section (response + hit)
tbl = tbl[['id', 'reference',
           'naive_response',   'naive_hit',
           'section_response', 'section_hit']]
tbl


## 8. RAGAS — score both runs

We use **gpt-4o as the judge** (different from the gpt-4o-mini answerer, per the M16 *"LLM-as-judge done responsibly"* slide). Five metrics:

- **Faithfulness** — every claim in the answer is supported by the retrieved context.
- **Response relevancy** — the answer is on-topic for the question.
- **Context precision (with reference)** — the gold excerpt appears in the right place in the retrieved chunks.
- **Context recall** — the gold answer's claims are supported by the retrieved chunks.
- **Factual correctness** — claim-level F1 between the model's answer and the gold reference. Tends to collapse to 0 on our terse references — see the comment in the code cell + the M16 slide for the mechanism and the `mode='precision'` workaround.

The unanswerable questions are dropped from RAGAS (no ground truth) — they're scored by abstention recall instead.


In [ ]:
# Silence RAGAS's deprecation warnings — we deliberately use the stable sync API
# which is simpler to reason about than the new async `collections` runner.
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    Faithfulness, ResponseRelevancy,
    LLMContextPrecisionWithReference, LLMContextRecall,
    FactualCorrectness,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

judge_llm = LangchainLLMWrapper(ChatOpenAI(model=JUDGE_MODEL, temperature=0))
judge_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

# FactualCorrectness knobs (defaults shown):
#   mode='f1' | 'precision' | 'recall'    — for terse references like ours ("60",
#                                            "EDULEARN24") switch to 'precision' to
#                                            avoid the recall-collapse-to-0 failure.
#   atomicity='low' | 'high'              — finer claim decomposition on long answers
#   coverage='low' | 'high'                — strictness of claim coverage
#   beta=1.0                               — F-β weight when mode='f1'
METRICS = [
    Faithfulness(),
    ResponseRelevancy(),
    LLMContextPrecisionWithReference(),
    LLMContextRecall(),
    FactualCorrectness(),   # try mode='precision' if scores collapse on short refs
]

def ragas_score(runs_df: pd.DataFrame) -> pd.DataFrame:
    answerable = runs_df[runs_df['type'] != 'unanswerable'].copy()
    ds = EvaluationDataset.from_list(answerable[
        ['user_input', 'response', 'retrieved_contexts', 'reference', 'reference_contexts']
    ].to_dict('records'))
    result = evaluate(dataset=ds, metrics=METRICS, llm=judge_llm, embeddings=judge_emb)
    out = result.to_pandas()
    out.insert(0, 'id', answerable['id'].values)
    return out

print("scoring naive ...")
naive_scores   = ragas_score(naive_runs)
print("scoring section ...")
section_scores = ragas_score(section_runs)

naive_scores.head()


## 9. Side-by-side summary

Mean per metric, naive vs section. The headline answer is in this one table — but inspect the per-question diff in the next section before committing to a verdict, because means hide where each chunker actually wins.


In [63]:
# The only numeric columns in scores_df are the metric scores — average them.
summary = pd.DataFrame({
    "naive":   naive_scores.select_dtypes("number").mean(),
    "section": section_scores.select_dtypes("number").mean(),
}).T.round(3)
summary


,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall,factual_correctness(mode=f1)
naive,0.867,0.505,0.65,0.733,0.027
section,0.867,0.603,0.75,0.800,0.027


## 10. Where does each chunker actually win?

Mean scores can hide everything. Here's the per-question diff for faithfulness — sorted by `|naive − section|` so the most informative rows come first.

Read the top of this table to find specific questions where the two chunkers disagree the most — those are the ones to inspect by hand.


In [66]:
def diff_df(naive_df, section_df, metric_col):
    merged = naive_df[['id', metric_col]].rename(columns={metric_col: 'naive'}) \
        .merge(section_df[['id', metric_col]].rename(columns={metric_col: 'section'}), on='id')
    merged['delta'] = (merged['section'] - merged['naive']).round(3)
    merged['naive']   = merged['naive'].round(3)
    merged['section'] = merged['section'].round(3)
    merged = merged.merge(qa[['id', 'question']], on='id')
    return merged.sort_values('delta', key=abs, ascending=False)

faith_diff = diff_df(naive_scores, section_scores, 'faithfulness')
faith_diff.head(10)


,id,naive,section,delta,question
0,mc-001,1.0,1.0,0.0,Which LLM does the paper use as a Scrum Master...
1,mc-002,1.0,1.0,0.0,How many student groups were used to evaluate ...
2,mc-003,1.0,1.0,0.0,How long are the daily standup meetings descri...
3,mc-004,1.0,1.0,0.0,For which obstacle category did GPT provide ze...
4,mc-005,1.0,1.0,0.0,What is the institutional affiliation of the p...
5,ow-001,1.0,1.0,0.0,What percentage of the total reported obstacle...
6,ow-002,1.0,1.0,0.0,How many total obstacles were reported across ...
7,ow-003,1.0,1.0,0.0,At which conference was this paper presented?
8,ow-004,1.0,1.0,0.0,How many primary roles does the paper describe...
9,ow-005,0.0,0.0,0.0,In which city was the EDULEARN24 conference held?


## 11. Abstention recall on the 5 unanswerable questions

For each chunker, what fraction of the 5 deliberately-unanswerable questions did the model correctly refuse? A clean refusal starts with *"I don't know"* (the prompt licences that exact phrase).

If both chunkers score 100 %, the abstention prompt is doing its job. If either drops below 100 %, the model is being misled by spuriously similar chunks — that's a finding worth showing students.


In [ ]:
def abstention_recall(runs_df: pd.DataFrame) -> dict:
    un = runs_df[runs_df['type'] == 'unanswerable']
    refused = un['response'].str.lower().str.startswith("i don't know") | \
              un['response'].str.lower().str.startswith("i dont know")
    return {
        'n_unanswerable': len(un),
        'n_refused':      int(refused.sum()),
        'recall':         float(refused.mean()) if len(un) else None,
    }

print("naive  :", abstention_recall(naive_runs))
print("section:", abstention_recall(section_runs))

# Show the actual answers so we can audit any that didn't refuse.
def show_unanswerable(runs_df, label):
    un = runs_df[runs_df['type'] == 'unanswerable'][['id', 'user_input', 'response']]
    print(f"\n=== {label} ===")
    for _, r in un.iterrows():
        print(f"  [{r['id']}] {r['user_input']}")
        print(f"      → {r['response']}")

show_unanswerable(naive_runs,   "naive answers on unanswerable Qs")
show_unanswerable(section_runs, "section answers on unanswerable Qs")


## 12. Worst-scoring spot-checks

For each metric, the lowest-scoring question across both runs. This is the pedagogical payoff — it puts a concrete failure in front of you for each abstract metric.


In [ ]:
def worst_example(scores_df, metric_col, label):
    # RAGAS `.to_pandas()` already includes user_input / response / retrieved_contexts
    # alongside the metric scores, so no merge is needed (and merging would clash on
    # those duplicate columns).
    if metric_col not in scores_df.columns:
        print(f"(metric {metric_col} not in results)")
        return
    row = scores_df.sort_values(metric_col).iloc[0]
    print(f"--- {label}  |  metric {metric_col} = {row[metric_col]:.3f} ---")
    print(f"Q: {row['user_input']}")
    print(f"A: {row['response'][:200]}")
    print(f"top-1 chunk preview: {row['retrieved_contexts'][0][:200]}…")
    print()

for m in naive_scores.select_dtypes("number").columns:
    worst_example(naive_scores,   m, "NAIVE")
    worst_example(section_scores, m, "SECTION")


## 13. Takeaways

Things to point out in class once the numbers are in:

- **Which chunker wins on which metric.** Section-aware *usually* wins on `context_recall` (it keeps a topical thread intact); naive *can* win on `context_precision` because smaller fixed-size chunks carry less filler in any single hit. But the ranking is corpus-dependent — don't over-generalise from one paper.
- **The per-question diff is where the lesson is.** Mean recall@k hides which kinds of questions each chunker helps with. The top of the diff table tells the story.
- **Abstention is on the prompt, not the chunker.** Both chunkers should refuse all 5 unanswerable questions if the prompt is well calibrated. If they don't, the prompt is the bug — not the retriever.
- **Judge-model independence matters.** We grade gpt-4o-mini answers with gpt-4o. Same model on both sides risks self-grading bias (callback to M16 slide 12).
- **The numbers are stochastic.** Re-run, expect ±0.05 per metric. Sample size n=15 (answerable questions) is small — treat differences smaller than ~0.10 with caution.

> The notebook is meant to be a **template**. Swap the two chunkers for any two stages you care about (different embedders, different reranker configs) and the same RAGAS + abstention scaffolding gives you a clean side-by-side.
